# NII Forecast 2025–2026 — Greek Systemic Banks

Simple NII forecasting model using the rate/volume decomposition framework from `03_analysis/04_nii_rate_volume_walk.ipynb`.

**Model**: `NII = NIM × Total Assets` → forecast NIM and Total Assets separately, multiply.

**ECB Rate Scenarios (as of May 2026 — ECB cutting cycle):**
| Scenario | 2025 MRO | 2026 MRO | Assumption |
|----------|----------|----------|------------|
| Base | 2.50% | 2.00% | Gradual cuts, normalise to ~2% neutral |
| Hawkish | 3.00% | 2.75% | Inflation resurgence, fewer cuts |
| Dovish | 2.00% | 1.50% | Rapid cuts, growth concerns |

**NIM Sensitivity**: NIM assumed to move ~45bps per 100bps ECB rate change (partial pass-through, based on historical NIM vs MRO regression over 2022–2024).

**Volume (Assets) Growth**: simple linear trend extrapolation from 2022–2024 CAGR per bank.

**Confidence Intervals**: 80% and 95% CI derived from historical NIM forecast residuals (2022–2024 in-sample).

> **Disclaimer**: This is a portfolio demonstration model, not investment advice. 3 data points (2022–2024) are
> insufficient for robust statistical inference. Treat CIs as illustrative ranges, not precise probability bounds.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

COLORS = {
    'Eurobank':    '#0067B1',
    'Alpha Bank':  '#E2231A',
    'Piraeus Bank':'#F7A600',
    'NBG':         '#003087',
}
BANKS = list(COLORS.keys())
DATA_DIR = Path('../02_Banking_Sector_Dashboard/data/processed')

kpis = pd.read_csv(DATA_DIR / 'kpis_final.csv')
print(f'Loaded {len(kpis)} bank-year rows.')
print('Historical NIM and Total Assets:')
print(kpis[['bank','year','nii','nim','total_assets']].to_string(index=False))

In [ ]:
# ── ECB rate scenarios and NIM pass-through ───────────────────────────────────

# ECB MRO historical (annual averages)
ECB_HIST = {2022: 0.31, 2023: 3.87, 2024: 3.53}

# Scenarios: (base, hawkish, dovish)
ECB_SCENARIOS = {
    'Base':     {2025: 2.50, 2026: 2.00},
    'Hawkish':  {2025: 3.00, 2026: 2.75},
    'Dovish':   {2025: 2.00, 2026: 1.50},
}
SCENARIO_COLORS = {'Base': '#FFFFFF', 'Hawkish': '#EF553B', 'Dovish': '#636EFA'}

# NIM sensitivity: 45bps NIM change per 100bps ECB rate change
NIM_SENSITIVITY = 0.45

# Fit per-bank NIM trend using 2022-2024 data
# NIM ~ alpha + beta × ECB_MRO
# Use OLS with 3 data points per bank

ecb_hist_vals = np.array([ECB_HIST[y] for y in [2022, 2023, 2024]])

bank_params = {}
for bank in BANKS:
    bdf = kpis[kpis['bank'] == bank].sort_values('year')
    nim_vals = bdf['nim'].values  # already in %
    # OLS: NIM = alpha + beta * ECB_rate
    slope, intercept, r, p, se = stats.linregress(ecb_hist_vals, nim_vals)
    residuals = nim_vals - (intercept + slope * ecb_hist_vals)
    rmse = np.std(residuals, ddof=1)
    bank_params[bank] = {'slope': slope, 'intercept': intercept, 'rmse': rmse, 'r2': r**2}
    print(f'{bank}: NIM = {intercept:.3f} + {slope:.3f} × ECB_MRO | R²={r**2:.3f} | RMSE={rmse:.3f}pp')

print(f'\nNote: R² based on only 3 data points — directional only, not statistically robust.')

In [ ]:
# ── Forecast total assets (linear trend extrapolation) ────────────────────────
FORECAST_YEARS = [2025, 2026]

asset_forecasts = {}
for bank in BANKS:
    bdf = kpis[kpis['bank'] == bank].sort_values('year')
    years = bdf['year'].values.astype(float)
    assets = bdf['total_assets'].values
    slope, intercept, r, p, se = stats.linregress(years, assets)
    forecasts = {yr: intercept + slope * yr for yr in FORECAST_YEARS}
    asset_forecasts[bank] = forecasts
    print(f'{bank} asset CAGR (linear trend): {slope:.0f}€m/yr | '
          f'2025F={forecasts[2025]:.0f}m | 2026F={forecasts[2026]:.0f}m')

print('\nNote: Eurobank 2024 asset jump (Hellenic Bank) inflates trend; treat 2025/2026 with caution.')

In [ ]:
# ── Build NII forecasts per bank × scenario × year ───────────────────────────
Z_80 = 1.282  # z-score for 80% CI
Z_95 = 1.960  # z-score for 95% CI

forecasts = []
for bank in BANKS:
    p = bank_params[bank]
    for scenario, ecb_path in ECB_SCENARIOS.items():
        for yr in FORECAST_YEARS:
            nim_f   = p['intercept'] + p['slope'] * ecb_path[yr]  # NIM forecast (%)
            assets_f = asset_forecasts[bank][yr]
            nii_f   = nim_f / 100 * assets_f  # NII = NIM × Assets
            # CI based on NIM RMSE propagated through NIM × Assets
            nim_se  = p['rmse']  # pp
            nii_se  = nim_se / 100 * assets_f
            forecasts.append({
                'bank': bank, 'scenario': scenario, 'year': yr,
                'nim_f': nim_f, 'assets_f': assets_f,
                'nii_f': nii_f,
                'nii_lo80': nii_f - Z_80 * nii_se,
                'nii_hi80': nii_f + Z_80 * nii_se,
                'nii_lo95': nii_f - Z_95 * nii_se,
                'nii_hi95': nii_f + Z_95 * nii_se,
            })

fcst = pd.DataFrame(forecasts)

print('\n── Base Scenario NII Forecast ──')
base = fcst[fcst['scenario']=='Base'][['bank','year','nii_f','nii_lo80','nii_hi80']].round(0)
print(base.to_string(index=False))

In [ ]:
# ── Chart 1: Fan Chart — NII Forecast 2025-2026 per Bank ──────────────────────
fig1 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'<b>{b}</b>' for b in BANKS],
    vertical_spacing=0.16, horizontal_spacing=0.10,
)

for bi, bank in enumerate(BANKS):
    row, col = divmod(bi, 2)
    bkpis = kpis[kpis['bank'] == bank].sort_values('year')

    # Historical line
    fig1.add_trace(go.Scatter(
        x=bkpis['year'].tolist(), y=bkpis['nii'].tolist(),
        mode='lines+markers', line=dict(color=COLORS[bank], width=2.5),
        marker=dict(size=8), name=f'{bank} Historical',
        showlegend=(bi == 0), legendgroup='hist',
    ), row=row+1, col=col+1)

    # Scenario lines
    for scenario in ECB_SCENARIOS:
        sc_fcst = fcst[(fcst['bank']==bank) & (fcst['scenario']==scenario)].sort_values('year')
        # Connect from 2024 actual to forecast
        x_all = [2024] + sc_fcst['year'].tolist()
        y_all = [bkpis[bkpis['year']==2024]['nii'].values[0]] + sc_fcst['nii_f'].tolist()

        fig1.add_trace(go.Scatter(
            x=x_all, y=y_all,
            mode='lines+markers',
            line=dict(color=SCENARIO_COLORS[scenario], width=1.8, dash='dot'),
            marker=dict(size=7),
            name=f'{scenario} scenario',
            showlegend=(bi == 0),
            legendgroup=scenario,
        ), row=row+1, col=col+1)

        if scenario == 'Base':
            # CI bands
            x_fcst = sc_fcst['year'].tolist()
            fig1.add_trace(go.Scatter(
                x=x_fcst + x_fcst[::-1],
                y=sc_fcst['nii_hi80'].tolist() + sc_fcst['nii_lo80'].tolist()[::-1],
                fill='toself', fillcolor='rgba(255,255,255,0.12)',
                line=dict(color='rgba(255,255,255,0)'),
                name='80% CI (Base)', showlegend=(bi == 0), legendgroup='ci80',
            ), row=row+1, col=col+1)
            fig1.add_trace(go.Scatter(
                x=x_fcst + x_fcst[::-1],
                y=sc_fcst['nii_hi95'].tolist() + sc_fcst['nii_lo95'].tolist()[::-1],
                fill='toself', fillcolor='rgba(255,255,255,0.06)',
                line=dict(color='rgba(255,255,255,0)'),
                name='95% CI (Base)', showlegend=(bi == 0), legendgroup='ci95',
            ), row=row+1, col=col+1)

    # Vertical separator at 2024
    fig1.add_vline(x=2024.5, line_dash='dash', line_color='rgba(255,255,255,0.2)',
                   row=row+1, col=col+1)

fig1.update_layout(
    title_text='<b>NII Forecast 2025–2026</b> | Base / Hawkish / Dovish ECB scenarios | Shaded = 80%/95% CI (Base)',
    title_font_size=15, template='plotly_dark', height=620,
    legend=dict(orientation='h', y=-0.16, x=0.5, xanchor='center', font_size=11),
    paper_bgcolor='#0f1117', plot_bgcolor='#0f1117',
)
fig1.update_yaxes(title_text='NII (€ million)')
fig1.show()

In [ ]:
# ── Chart 2: Sector NII by Scenario ───────────────────────────────────────────
sector_hist = kpis.groupby('year')['nii'].sum().reset_index()
sector_fcst = fcst.groupby(['scenario','year'])[['nii_f','nii_lo80','nii_hi80','nii_lo95','nii_hi95']].sum().reset_index()

fig2 = go.Figure()
# Historical
fig2.add_trace(go.Scatter(
    x=sector_hist['year'], y=sector_hist['nii'],
    mode='lines+markers', line=dict(color='white', width=3),
    marker=dict(size=10), name='Historical sector NII',
))

for scenario in ECB_SCENARIOS:
    sc = sector_fcst[sector_fcst['scenario']==scenario].sort_values('year')
    # Bridge from 2024 actual
    x_all = [2024] + sc['year'].tolist()
    y_all = [sector_hist[sector_hist['year']==2024]['nii'].values[0]] + sc['nii_f'].tolist()
    fig2.add_trace(go.Scatter(
        x=x_all, y=y_all,
        mode='lines+markers+text',
        line=dict(color=SCENARIO_COLORS[scenario], width=2.5, dash='dot'),
        marker=dict(size=9),
        text=[None] + [f'{v/1000:.1f}bn' for v in sc['nii_f'].tolist()],
        textposition='top center',
        name=f'{scenario} scenario',
    ))

fig2.add_vline(x=2024.5, line_dash='dash', line_color='rgba(255,255,255,0.25)')
fig2.add_annotation(x=2024.6, y=sector_hist['nii'].max()*0.9,
    text='Forecast →', showarrow=False, font=dict(color='white', size=12))

fig2.update_layout(
    title_text='<b>4-Bank Sector NII Forecast</b> | 2025–2026 by ECB scenario (€ million)',
    template='plotly_dark', height=400,
    yaxis_title='Sector NII (€ million)',
    legend=dict(orientation='h', y=-0.18, x=0.5, xanchor='center'),
    paper_bgcolor='#0f1117', plot_bgcolor='#0f1117',
)
fig2.show()

In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────────
print('\n── NII Forecast Summary (€ million) ──')
pivot = fcst[fcst['year']==2026].pivot_table(index='bank', columns='scenario', values='nii_f').round(0)
pivot['Historical_2024'] = kpis[kpis['year']==2024].set_index('bank')['nii']
pivot = pivot[['Historical_2024', 'Dovish', 'Base', 'Hawkish']]
print('2026 NII Forecast vs 2024 Actual:')
print(pivot.to_string())

print('\nECB Rate Assumptions:')
for sc, path in ECB_SCENARIOS.items():
    print(f'  {sc}: 2025={path[2025]:.2f}%, 2026={path[2026]:.2f}%')

In [ ]:
# ── Final validation ──────────────────────────────────────────────────────────
assert len(fcst) == len(BANKS) * len(ECB_SCENARIOS) * len(FORECAST_YEARS)
assert fcst['nii_f'].gt(0).all(), 'All NII forecasts must be positive'
assert (fcst['nii_hi80'] > fcst['nii_lo80']).all(), 'CI upper must exceed lower'
assert (fcst['nii_hi95'] > fcst['nii_hi80']).all(), '95% CI must be wider than 80%'

# Hawkish scenario should give higher NII than Dovish for all banks in 2026
hawk_nii = fcst[(fcst['scenario']=='Hawkish') & (fcst['year']==2026)].set_index('bank')['nii_f']
dove_nii = fcst[(fcst['scenario']=='Dovish')  & (fcst['year']==2026)].set_index('bank')['nii_f']
for bank in BANKS:
    if bank in hawk_nii.index and bank in dove_nii.index:
        assert hawk_nii[bank] > dove_nii[bank], f'{bank}: Hawkish NII should exceed Dovish'

print('✅ All checks passed.')
base_sector_2026 = fcst[(fcst['scenario']=='Base') & (fcst['year']==2026)]['nii_f'].sum()
hist_sector_2024 = kpis[kpis['year']==2024]['nii'].sum()
print(f'   Base scenario sector NII 2026: {base_sector_2026:.0f}m (vs {hist_sector_2024:.0f}m in 2024)')
print(f'   Implied sector NII change: {(base_sector_2026/hist_sector_2024-1)*100:+.1f}%')